In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "3"

In [ ]:
import torch
free, total = torch.cuda.mem_get_info()
print(f"free={free/1e9:.1f} GB  total={total/1e9:.1f} GB")
print(torch.cuda.memory_summary())

In [ ]:
import sys
from pathlib import Path

repo_root = Path.cwd().parent
sys.path.insert(0, str(repo_root))

In [ ]:
import io
from pathlib import Path
import yaml

import numpy as np
import pyarrow.parquet as pq
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

from dataloader import patient_id_from_relpath, patient_in_val

In [ ]:
cfg = yaml.safe_load(Path(f'{repo_root}/configs/main.yaml').read_text())


In [ ]:

def clean_transform():
    return transforms.Compose([transforms.Resize((224, 224), antialias=True), transforms.ToTensor()])


class TCGACleanTileDataset(Dataset):
    # Build a (shard_idx, row_in_shard) index over the requested patient split and
    # return clean images with barcodes. `is_train=None` keeps every patient;
    # True/False select the same train / held-out slices as dataloader.TCGATileDataset.
    def __init__(self, cfg, is_train=None, transform=None):
        data = cfg["data"]
        dataset_dir = Path(data["dataset_dir"])
        self.shards = sorted(dataset_dir.glob("shard-*.parquet"))
        if not self.shards:
            raise FileNotFoundError(f"No parquet shards (shard-*.parquet) under {dataset_dir}.")
        self.transform = transform if transform is not None else clean_transform()
        # Lazy per-worker ParquetFile handles so fork-children own their file positions.
        self._readers = [None] * len(self.shards)
        in_split_shard, in_split_row = [], []
        for shard_idx, shard_path in enumerate(self.shards):
            paths = pq.read_table(str(shard_path), columns=["path"], memory_map=True)["path"].to_pylist()
            for row_idx, p in enumerate(paths):
                if is_train is not None and patient_in_val(patient_id_from_relpath(p), data["split_seed"], data["val_fraction"]) == is_train:
                    continue
                in_split_shard.append(shard_idx)
                in_split_row.append(row_idx)
        if not in_split_shard:
            raise ValueError(f"no tiles found in {dataset_dir} for is_train={is_train}")
        self.shard_of = np.asarray(in_split_shard, dtype=np.int32)
        self.row_of = np.asarray(in_split_row, dtype=np.int32)

    def __len__(self):
        return int(self.shard_of.shape[0])

    def __getitem__(self, idx):
        idx = int(idx)
        shard_idx = int(self.shard_of[idx])
        row_idx = int(self.row_of[idx])
        reader = self._readers[shard_idx]
        if reader is None:
            reader = pq.ParquetFile(str(self.shards[shard_idx]), memory_map=True)
            self._readers[shard_idx] = reader
        rg_size = reader.metadata.row_group(0).num_rows
        table = reader.read_row_group(row_idx // rg_size, columns=["path", "jpeg"])
        rel = table["path"][row_idx % rg_size].as_py()
        jpeg_bytes = table["jpeg"][row_idx % rg_size].as_py()
        with Image.open(io.BytesIO(jpeg_bytes)) as img:
            image = self.transform(img.convert("RGB"))
        slide_stem = rel.split("/", 1)[0]
        return {
            "image": image,
            "patient_id": "-".join(slide_stem.split("-")[:3]),
            "slide_id": slide_stem,
            "sample_idx": idx,
        }


In [ ]:
ds = TCGACleanTileDataset(cfg, is_train=False)

In [ ]:
loader = DataLoader(ds, batch_size=512, num_workers=4, shuffle=True, pin_memory=True)

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

from model import DinoV2ViT, PrototypeHead, newton_schulz


def load_nanopath_model(checkpoint_path, device="cuda", weights="ema"):
    """weights: 'ema' -> ckpt['model_ema'] (teacher, usual for eval)
                'model' -> ckpt['model'] (student). Returns (model, cfg)."""
    state_key = {"ema": "model_ema", "model": "model"}[weights]
    ckpt = torch.load(checkpoint_path, map_location="cpu", weights_only=False)  # payload has the config dict
    cfg = ckpt["config"]
    model = DinoV2ViT(variant=cfg["model"]["type"])
    model.load_state_dict(ckpt[state_key], strict=True)
    model.to(device).eval()
    for p in model.parameters():
        p.requires_grad = False
    return model, cfg

def load_prototype_head(checkpoint_path, device="cuda", weights="ema"):
    state_key = {"ema": "dino_head_ema", "model": "dino_head"}[weights]
    ckpt = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
    if state_key not in ckpt:
        raise KeyError(f"{state_key!r} not in checkpoint — heads are only in full checkpoints (latest.pt).")
    h = ckpt["config"]["prototype_head"]
    head_state = ckpt[state_key]
    in_dim = head_state.get("mlp.0.weight", head_state.get("mlp.weight")).shape[1]  # backbone embed_dim
    head = PrototypeHead(in_dim, h["n_prototypes"], h["hidden_dim"], h["prototype_dim"],
                         h["n_layers"], h["ns_steps"], h["orthogonal"])
    head.load_state_dict(head_state, strict=True)
    return head.to(device).eval()

@torch.no_grad()
def prototype_similarity(head, embedding, kind="cosine"):
    """embedding: model.probe_features(...) output, [in_dim] or [B, in_dim].
       kind: 'cosine' [-1,1] (default) | 'logit' (temp-scaled) | 'prob' (softmax)."""
    head.eval()
    p = next(head.parameters())
    x = embedding.to(p.device, dtype=p.dtype)
    squeeze = x.dim() == 1
    if squeeze:
        x = x.unsqueeze(0)
    x = F.normalize(head.mlp(x), dim=-1, p=2)                                   # into prototype space
    prototypes = newton_schulz(head.prototypes, steps=head.ns_steps) if head.orthogonal \
                 else F.normalize(head.prototypes, dim=-1, p=2)                 # the bank the head scores against
    cos = x @ prototypes.T
    scale = head.logit_scale.clamp(max=math.log(100)).exp()
    out = {"cosine": cos, "logit": scale * cos, "prob": (scale * cos).softmax(dim=-1)}[kind]
    return out.squeeze(0) if squeeze else out


In [ ]:
CKPT = "/data/raymond.biju/nanopath/20260703_op1_skN_swdN_entregY_orthoY_skibotN/latest.pt"
device = "cuda"

model, cfg = load_nanopath_model(CKPT, device, weights="ema")   # backbone <- model_ema
head = load_prototype_head(CKPT, device, weights="ema")   # head     <- dino_head_ema

mean = torch.tensor(cfg["data"]["mean"], device=device).view(1, 3, 1, 1)
std  = torch.tensor(cfg["data"]["std"],  device=device).view(1, 3, 1, 1)

In [ ]:
proto_feats, patient_ids = [], []
with torch.no_grad():
    for i, batch in enumerate(loader):
        if i >= 5:
            break
        x = batch["image"].to(device, non_blocking=True)
        with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
            emb = model.probe_features((x - mean) / std)
        sims = prototype_similarity(head, emb.float())
        proto_feats.append(sims.cpu().numpy())
        patient_ids.extend(batch["patient_id"])

proto_feats = np.concatenate(proto_feats, axis=0)
patient_ids = np.array(patient_ids)
print(proto_feats.shape, patient_ids.shape)


In [ ]:
import numpy as np
import pandas as pd

meta = pd.read_csv('cbio_all_clinical.csv')

PATIENT_COL = "patient_id"
SITE_COL    = "SITE_OF_TUMOR_TISSUE"

# Normalize the CSV key to the 3-part patient barcode so it matches our patient_ids
# (handles CSVs that store longer sample/aliquot barcodes).
key = meta[PATIENT_COL].astype(str).str.split("-").str[:3].str.join("-")
site_lookup = dict(zip(key, meta[SITE_COL]))

sites = np.array([str(site_lookup.get(pid, "Unknown")) for pid in patient_ids], dtype=object)

# SANITY CHECK — if this is mostly "Unknown", PATIENT_COL is wrong.
matched = int((sites != "Unknown").sum())
print(f"matched {matched}/{len(sites)} tiles to a site")
print(pd.Series(sites).value_counts().head(12))

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm

MIN_TILES = 10

def short_site(s):
    return s.rsplit("-", 1)[-1].strip() if "-" in s else s   # part after the last hyphen

# 1. site x prototype mean-activation matrix (drop Unknown + sparse sites)
df = pd.DataFrame(proto_feats)
df["site"] = sites
df = df[df["site"] != "Unknown"]
keep = df["site"].value_counts()
keep = keep[keep >= MIN_TILES].index
df = df[df["site"].isin(keep)]
assert len(keep) >= 2, f"only {len(keep)} site(s) with >={MIN_TILES} tiles — raise N_BATCHES"

M = df.groupby("site").mean()                       # sites x prototypes
M = M.loc[M.mean(axis=1).sort_values().index]       # stable site order
Mc = M.subtract(M.mean(axis=0), axis=1)             # center per prototype -> selectivity

# 2. transpose -> prototypes (rows) x sites (cols); group prototype rows by most-selective site
MT = Mc.T
row_order = np.argsort(np.argmax(MT.values, axis=1), kind="stable")
MT = MT.iloc[row_order]

# 3. diverging heatmap: blue (below-avg) <-> gray 0 <-> red (above-avg), symmetric
FS = 22   # base font size — turn this up/down to taste

vmax = float(np.abs(MT.values).max())
cmap = LinearSegmentedColormap.from_list("blue_gray_red", ["#2a78d6", "#f0efec", "#e34948"])
norm = TwoSlopeNorm(vcenter=0.0, vmin=-vmax, vmax=vmax)

fig, ax = plt.subplots(figsize=(0.9 * MT.shape[1] + 4, 11), dpi=130)
fig.patch.set_facecolor("white")
im = ax.imshow(MT.values, aspect="auto", cmap=cmap, norm=norm, interpolation="nearest")

ax.set_xticks(range(MT.shape[1]))
ax.set_xticklabels([short_site(s) for s in MT.columns], rotation=45, ha="right", fontsize=FS, color="#0b0b0b")
ax.set_yticks([])
ax.set_ylabel(f"prototype  (n={MT.shape[0]})", color="#52514e", fontsize=FS)
ax.set_xlabel("site of tumor tissue", color="#52514e", fontsize=FS * 1.1)
ax.set_title("Prototype activation by tumor site", color="#0b0b0b", fontsize=FS * 1.25, pad=14)
for sp in ax.spines.values(): sp.set_visible(False)

cbar = fig.colorbar(im, ax=ax, fraction=0.035, pad=0.02)
cbar.set_label("mean cosine sim − prototype avg", color="#52514e", fontsize=FS)
cbar.ax.tick_params(labelsize=FS * 0.8, colors="#52514e")
cbar.outline.set_visible(False)

plt.tight_layout()
fig.savefig("heatmap.png", dpi=200, bbox_inches="tight")   # bigger fonts persist into the file you \includegraphics
plt.show()


